# 🏠 Dự Đoán Giá Nhà Hà Nội — Linear Regression
**Nguồn dữ liệu:** [Vietnam Housing Dataset - Hanoi (Kaggle)](https://www.kaggle.com/datasets/ladcva/vietnam-housing-dataset-hanoi)  
**Mục tiêu:** Xây dựng mô hình hồi quy tuyến tính dự đoán `Giá nhà` dựa trên các đặc trưng như diện tích, số phòng, và toạ độ địa lý.

---
## Mục Lục
1. Lý Thuyết Linear Regression
2. Cài Đặt & Import Thư Viện
3. Tải & Khám Phá Dữ Liệu (EDA)
   - *Bao gồm Trích xuất Kinh độ/Vĩ độ Tự động (Geocoding)*
4. Tiền Xử Lý Dữ Liệu
5. Xây Dựng & Huấn Luyện Mô Hình
6. Đánh Giá Mô Hình
7. Trực Quan Hóa Loss & Dự Đoán
8. Lưu & Tải Mô Hình
9. Kết Luận


## 1. Lý Thuyết Linear Regression

### 1.1 Định Nghĩa
**Hồi quy tuyến tính (Linear Regression)** là thuật toán học có giám sát dự đoán giá trị liên tục, giả định mối quan hệ tuyến tính giữa $X$ và $y$.

### 1.2 Hàm Giả Thuyết
$$\hat{y} = \theta_0 + \theta_1 x_1 + \cdots + \theta_n x_n = \mathbf{X}\boldsymbol{\theta}$$

### 1.3 Hàm Mất Mát — MSE
$$\mathcal{L} = \frac{1}{m}\sum_{i=1}^{m}(\hat{y}^{(i)} - y^{(i)})^2$$

### 1.4 Gradient Descent
$$\theta_j := \theta_j - \alpha \cdot \frac{\partial \mathcal{L}}{\partial \theta_j}, \quad \frac{\partial \mathcal{L}}{\partial \theta_j} = \frac{2}{m}\sum_{i=1}^{m}(\hat{y}^{(i)} - y^{(i)})x_j^{(i)}$$

### 1.5 Normal Equation (Nghiệm Giải Tích)
$$\boldsymbol{\theta}^* = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

### 1.6 Các Chỉ Số Đánh Giá
| Chỉ số | Công thức | Ý nghĩa |
|--------|-----------|----------|
| MSE | $\frac{1}{m}\sum(\hat{y}-y)^2$ | Sai số bình phương trung bình |
| RMSE | $\sqrt{MSE}$ | Sai số cùng đơn vị với $y$ |
| MAE | $\frac{1}{m}\sum|\hat{y}-y|$ | Sai số tuyệt đối trung bình |
| $R^2$ | $1-\frac{SS_{res}}{SS_{tot}}$ | Tỷ lệ phương sai giải thích được |


## 2. Import Thư Viện
*Cài đặt thêm thư viện `geopy` để chuyển đổi địa chỉ thành kinh độ và vĩ độ.*
`!pip install geopy` (chạy command này trong Terminal nếu bạn chưa cài đặt).


In [ ]:
# === IMPORTS & SETUP ===
import os
import warnings
import json
import joblib
import time
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

warnings.filterwarnings('ignore')
np.random.seed(42)

# Tạo thư mục output
os.makedirs('plots',  exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data',   exist_ok=True)

# Style vẽ biểu đồ
plt.rcParams.update({
    'font.family': 'Segoe UI',
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#aaa',
    'ytick.color':      '#aaa',
    'text.color':       '#e0e0e0',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
})

print('Thư viện import thành công!')


## 3. Tải & Khám Phá Dữ Liệu (EDA)

### 3.1 Nguồn Dữ Liệu
Dataset Giá nhà Hà Nội, Việt Nam. Dữ liệu bao gồm các thông tin về diện tích, số tầng, số phòng ngủ, địa chỉ, quận huyện, và Mức giá.

| Cột | Mô tả |
|-----|-------|
| `Ngày` | Ngày đăng tin |
| `Địa chỉ` | Địa chỉ cụ thể (thường là tên phường, đường) |
| `Quận` | Quận (ví dụ: Đống Đa, Cầu Giấy,...) |
| `Huyện` | (Thường bị bỏ trống hoặc chung cột với Quận) |
| `Loại hình nhà ở` | Phân loại bất động sản |
| `Giấy tờ pháp lý` | Sổ đỏ, sổ hồng... |
| `Số tầng` | Chiều cao nhà |
| `Số phòng ngủ` | Tổng số phòng ngủ |
| `Diện tích` | Diện tích đất hoặc sử dụng |
| `Theo độ dài/ Rộng` | Kích thước MT/Ngang |
| `Giá/m2` | Giá tính theo mét vuông |
| `Giá` | **Giá nhà tổng (TARGET)** |


In [ ]:
# Tải dữ liệu từ tệp cục bộ đã download qua Kaggle
DATA_PATH = r"C:\Users\minht\.cache\kagglehub\datasets\ladcva\vietnam-housing-dataset-hanoi\versions\1\VN_housing_dataset.csv"

# Nếu bạn chạy ở máy khác, đưa file csv vào thư mục data/VN_housing_dataset.csv
if os.path.exists(DATA_PATH):
    df_raw = pd.read_csv(DATA_PATH)
else:
    print("Vui lòng cập nhật DATA_PATH trỏ đến file dữ liệu của bạn.")
    df_raw = pd.DataFrame() # Fallback

print(f'Shape kích thước dữ liệu: {df_raw.shape}')
df_raw.head()


In [ ]:
print('=== Thông tin dữ liệu ban đầu ===')
df_raw.info()


### 3.2 Tiền Xử Lý Cơ Bản: Trích Xuất Giá Trị `Giá` và Phân Phối
Đặc thù của dữ liệu nhà đất Việt Nam là giá được ghi dưới dạng chuỗi văn bản (VD: "3.5 triệu/m2", "800 triệu/m2", "Thỏa thuận").
Chúng ta cần viết một hàm bóc tách văn bản này thành dạng số nguyên (VND) sau đó nhân với Diện Tích để ra Giá Dòng.


In [ ]:
def parse_price(price_per_m2_str, area_val):
    if pd.isna(price_per_m2_str) or pd.isna(area_val):
        return np.nan
    price_str = str(price_per_m2_str).lower().strip()
    if price_str == 'thỏa thuận' or price_str == 'không xác định':
        return np.nan
    
    try:
        if 'triệu/m' in price_str:
            val = float(re.sub(r'[^\d.]', '', price_str.replace(',', '.')))
            return val * 1e6 * area_val
        elif 'nghìn/m' in price_str:
            val = float(re.sub(r'[^\d.]', '', price_str.replace(',', '.')))
            return val * 1e3 * area_val
        elif 'tỷ' in price_str and 'triệu' in price_str:
             parts = price_str.split('tỷ')
             ty_part = float(parts[0].strip().replace(',', '.'))
             trieu_part = float(re.sub(r'[^\d.]', '', parts[1].strip().replace(',', '.')))
             return ty_part * 1e9 + trieu_part * 1e6
        elif 'tỷ' in price_str:
             val = float(re.sub(r'[^\d.]', '', price_str.replace(',', '.')))
             return val * 1e9
        elif 'triệu' in price_str:
             val = float(re.sub(r'[^\d.]', '', price_str.replace(',', '.')))
             return val * 1e6
        else:
             val = float(re.sub(r'[^\d.]', '', price_str.replace(',', '.')))
             return val
    except Exception:
        return np.nan

def parse_area(area_str):
    if not isinstance(area_str, str): 
        return np.nan
    try: 
        return float(re.sub(r'[^\d.]', '', str(area_str).replace(',', '.')))
    except: 
        return np.nan

# Copy data tránh sửa trực tiếp
df = df_raw.copy()
df['area'] = df['Diện tích'].apply(parse_area)
df['target_price'] = df.apply(lambda row: parse_price(row['Giá/m2'], row['area']), axis=1)

# Loại bỏ những dòng không parse được giá hoặc giá bất hợp lý (ngoài mức 100tr - 500 tỷ)
df = df.dropna(subset=['target_price', 'area'])
df = df[(df['target_price'] >= 1e8) & (df['target_price'] <= 500e9)].copy()

# Lọc các nhà diện tích hợp lệ
df = df[(df['area'] >= 10) & (df['area'] <= 2000)].copy()

print(f"Số lượng mẫu tin hiện còn sau khi làm sạch Giá: {len(df)}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phân phối Mức Giá Nhà', fontsize=15, fontweight='bold', color='#00d4ff')

axes[0].hist(df['target_price'], bins=50, color='#00d4ff', alpha=0.8, edgecolor='#0a0a1a')
axes[0].set_title('Biểu đồ Tần suất (Histogram)')
axes[0].set_xlabel('Giá nhà (VND)')
axes[0].set_ylabel('Tần suất')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:,.0f} Tỷ'))
axes[0].grid(True)

axes[1].hist(np.log1p(df['target_price']), bins=50, color='#ff6b6b', alpha=0.8, edgecolor='#0a0a1a')
axes[1].set_title('Biểu đồ Tần suất (Thang Logarit)')
axes[1].set_xlabel('log(Giá nhà + 1)')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('plots/01_vietnam_target.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Giá nhà trung vị: {df["target_price"].median()/1e9:,.2f} Tỷ VNĐ')
print(f'Giá nhà trung bình: {df["target_price"].mean()/1e9:,.2f} Tỷ VNĐ')


### 3.3 Cấu trúc vị trí (Bản Đồ Phân Bố Geo)
*Sử dụng thư viện API OpenStreetMap (Geopy) để chuyển Địa chỉ Chuỗi thành Kinh độ & Vĩ độ*
Do dữ liệu có hơn 80.000 dòng, gọi API cho từng dòng sẽ rất lâu và bị khóa IP. Do đó, chúng ta sẽ **gom nhóm các quận/phường (Unique Tiers)** để truy vấn, sau đó Map ngược lại.


In [ ]:
# Gom danh sách địa chỉ. Chấp nhận sử dụng "Cột Phường/Địa chỉ" + "Quận"
df['Quận'] = df['Quận'].fillna('').astype(str).str.strip()
df['Địa chỉ'] = df['Địa chỉ'].fillna('').astype(str).str.strip()

# Trích xuất tên phường từ cột địa chỉ (nếu có cụm từ chứa 'Phường')
def extract_phuong(addr_str):
    if 'Phường' in addr_str:
        # Lấy từ "Phường" đến dấu phẩy hoặc gạch ngang gần nhất
        match = re.search(r'(Phường\s+[^,-]+)', addr_str)
        if match:
            return match.group(1).strip()
    return ""

df['Phuong_Clean'] = df['Địa chỉ'].apply(extract_phuong)
# Chuỗi Address Query
df['Query_Addr'] = df.apply(lambda row: f"{row['Phuong_Clean']}, {row['Quận']}, Hà Nội" if row['Phuong_Clean'] else f"{row['Quận']}, Hà Nội", axis=1)

unique_addrs = df['Query_Addr'].unique()
print(f"Tổng số phân lớp địa chỉ cần query: {len(unique_addrs)}")

# Cache lookup để không bị mất kết quả khi restart kernel
CACHE_FILE = 'data/hanoi_location_cache.json'
geo_cache = {}
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'r', encoding='utf-8') as f:
        geo_cache = json.load(f)

# Khởi tạo Geocoder với user_agent 
geolocator = Nominatim(user_agent="hanoi_housing_predictor_01")

print("Đang truy vấn toạ độ, xin chờ...")
new_found = 0
for addr in unique_addrs:
    if addr not in geo_cache:
        try:
            location = geolocator.geocode(addr, timeout=10)
            if location:
                geo_cache[addr] = {'lat': location.latitude, 'lng': location.longitude}
            else:
                geo_cache[addr] = None # Không tìm thấy
            time.sleep(1) # Bắt buộc delay 1s theo chính sách Nominatim OSM
            new_found += 1
            if new_found % 10 == 0:
                print(f" Đã quét được {new_found} địa chỉ mới...")
        except GeocoderTimedOut:
            print(f" Timeout cho: {addr}")
            continue

# Lưu cache mới
with open(CACHE_FILE, 'w', encoding='utf-8') as f:
    json.dump(geo_cache, f, ensure_ascii=False, indent=4)
print("Hoàn tất map toạ độ vào Cache!")

# Map dữ liệu vào DataFrame
DEFAULT_LAT, DEFAULT_LNG = 21.0285, 105.8048 # Toạ độ trung tâm Hà NộiFallback
def get_lat(addr):
    cache = geo_cache.get(addr)
    return cache['lat'] + np.random.uniform(-0.015, 0.015) if cache else DEFAULT_LAT + np.random.uniform(-0.05, 0.05)

def get_lng(addr):
    cache = geo_cache.get(addr)
    return cache['lng'] + np.random.uniform(-0.015, 0.015) if cache else DEFAULT_LNG + np.random.uniform(-0.05, 0.05)

df['latitude'] = df['Query_Addr'].apply(get_lat)
df['longitude'] = df['Query_Addr'].apply(get_lng)


In [ ]:
plt.figure(figsize=(10,7))
scatter = plt.scatter(df['longitude'], df['latitude'], alpha=0.4,
            s=df['target_price']/10e9, label='Nhà ở', 
            c=df['target_price']/1e9, cmap=plt.get_cmap('jet'))

plt.colorbar(scatter, label='Mức Giá Bán (Tỷ VND)')
plt.xlabel("Kinh Độ")
plt.ylabel("Vĩ Độ")
plt.title("Bản Đồ Phân Bố Giá Nhà Tại Hà Nội", color="#00d4ff", fontweight="bold")
plt.grid(True)
plt.savefig('plots/02_hanoi_map.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Tiền Xử Lý Dữ Liệu
Bóc tách các string còn lại: "Số phòng ngủ", "Số tầng"...


In [ ]:
def parse_rooms(room_str):
    if pd.isna(room_str): 
        return 0
    try:
        if isinstance(room_str, str): 
            return int(float(re.sub(r'[^\d.]', '', room_str)))
        return int(room_str)
    except: 
        return 0

df['bedrooms'] = df['Số phòng ngủ'].apply(parse_rooms)
df['floors'] = df['Số tầng'].apply(parse_rooms)
if 'Số toilet' in df.columns:
    df['bathrooms'] = df['Số toilet'].apply(parse_rooms)
else:
    df['bathrooms'] = df['bedrooms'] # Giả định số toilet tương đương nếu bị thiếu

# Lấy các cột quan trọng
model_df = df[['Loại hình nhà ở', 'area', 'floors', 'bedrooms', 'bathrooms', 'latitude', 'longitude', 'target_price']].copy()
model_df.rename(columns={'Loại hình nhà ở': 'house_type'}, inplace=True)

# Lọc bỏ giá trị thiếu (dropna)
model_df = model_df.dropna()
print(f"Kích thước tệp train cuối cùng: {model_df.shape}")


### 4.2 Chia Tập Dữ Liệu & Mã hóa
Chúng ta sử dụng `StandardScaler` cho biến số liên tục (area, coord) và `OneHotEncoder` cho loại nhà phố. Đây sẽ là bước quan trọng giúp Gradient Descent không bị chệch.


In [ ]:
# Chia X, y
X = model_df.drop(columns=['target_price'])
y = model_df['target_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Định nghĩa các pipeline con
numeric_features = ['area', 'floors', 'bedrooms', 'bathrooms', 'latitude', 'longitude']
categorical_features = ['house_type']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])


## 5. Xây Dựng & Huấn Luyện Mô Hình (Pipeline)


In [ ]:
# Lắp ghép toàn bộ qua 1 Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Bắt đầu tự động fit
pipeline.fit(X_train, y_train)
print("Huấn luyện Linear Regression giải tích (Normal Equation) hoàn tất!")


## 6. Đánh Giá Mô Hình
Tính toán MSE, RMSE và R² Test.


In [ ]:
def calculate_metrics(model, X_data, y_true):
    y_pred = model.predict(X_data)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

mse_train, rmse_train, mae_train, r2_train = calculate_metrics(pipeline, X_train, y_train)
mse_test, rmse_test, mae_test, r2_test = calculate_metrics(pipeline, X_test, y_test)

print(f"--- ĐÁNH GIÁ TẬP TRAIN ---")
print(f"RMSE: {rmse_train/1e9:,.2f} Tỷ VNĐ")
print(f"R2  : {r2_train:.4f}")

print(f"\n--- ĐÁNH GIÁ TẬP TEST ---")
print(f"RMSE: {rmse_test/1e9:,.2f} Tỷ VNĐ")
print(f"MAE : {mae_test/1e9:,.2f} Tỷ VNĐ")
print(f"R2  : {r2_test:.4f}")


## 7. Trực Quan Hóa Loss & Dự Đoán
### 7.1 Learning Curve Bằng SGDRegressor
Giải thuật Giải tích trực tiếp ra nghiệm (LinearRegression) ngay lập tức nên không có Lịch sử MSE.
Để vẽ biểu đồ MSE giảm dần, ta sử dụng Stochastic Gradient Descent.


In [ ]:
# Xử lý Transform data sẵn để đẩy vào vòng lặp Epochs SGD
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

sgd_reg = SGDRegressor(max_iter=1, tol=None, warm_start=True, penalty=None,
                       learning_rate="constant", eta0=0.005, random_state=42)

n_epochs = 150
train_errors, val_errors = [], []

y_train_arr = y_train.values / 1e9 # Scale xuống đơn vị ngàn tỷ để MSE không bị tràn vượt quá Int(64)
y_test_arr = y_test.values / 1e9

for epoch in range(n_epochs):
    sgd_reg.fit(X_train_processed, y_train_arr)
    
    y_train_predict = sgd_reg.predict(X_train_processed)
    y_val_predict = sgd_reg.predict(X_test_processed)
    
    train_errors.append(mean_squared_error(y_train_arr, y_train_predict))
    val_errors.append(mean_squared_error(y_test_arr, y_val_predict))

plt.figure(figsize=(10,5))
plt.plot(train_errors, "r-+", linewidth=2, label="Tập Train")
plt.plot(val_errors, "b-", linewidth=3, label="Tập Validation (Test)")
plt.legend(loc="upper right", fontsize=14)
plt.xlabel("Số Vòng Lặp (Epochs)", fontsize=14)
plt.ylabel("RMSE (Tỷ VNĐ^2)", fontsize=14)
plt.title("Đường Cong Học Tập (Learning Curve)", color="#00d4ff", fontweight="bold")
plt.grid(True)
plt.savefig('plots/03_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
y_pred = pipeline.predict(X_test)

# So sánh 100 sample đầu tiên
plt.figure(figsize=(10,6))
plt.plot(y_test.values[:100]/1e9, 'b.-', label='Giá Thực tế', markersize=8)
plt.plot(y_pred[:100]/1e9, 'rX--', label='Giá Dự đoán', alpha=0.7)
plt.ylabel("Giá Nhà (Tỷ VNĐ)")
plt.xlabel("Chỉ mục mẫu (100 mẫu)")
plt.title("Đường Thực Tế vs Dự Đoán (Mô hình Hà Nội)", color="#00d4ff")
plt.legend()
plt.grid(True)
plt.savefig('plots/04_real_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Lưu & Tải Mô Hình
Sau khi hoàn thiện mô hình Tốt Nhất trên tập dữ liệu Việt Nam, sử dụng Joblib xuất File để kết nối vào Django Backend của Ứng dụng Web.


In [ ]:
os.makedirs('models', exist_ok=True)
model_path = 'models/hanoi_model.pkl'

joblib.dump(pipeline, model_path)
print(f"Đã lưu mô hình Pipeline tại: {os.path.abspath(model_path)}")


## 9. Kết Luận
- Quá trình chuyển đổi toạ độ (*Geocoding*) bằng Geopy hoạt động để thiết lập toạ độ không gian.
- Mô hình Pipeline sẽ Tự Động chuẩn hóa (Standard Scaler) & Decode (OneHot) đối với mọi Yêu Cầu Gửi qua Giao diện Front-end, và đưa ra Giá Dự Đoán Trực Tiếp Mức Tỷ Đồng.
- Biểu đồ **Learning Curve** hội tụ ổn định, thể hiện quá trình Gradient Descent tránh được tình trạng Tràn Lỗi do dữ liệu đã được Preprocess chuẩn. 
